# HW7: Reinforcement Learning — PPO Balances the Cart-Pole
**NTUT MuJoCo Course — Lecture 8 · 30 points + 5 bonus**

Lectures 4 and 7 balanced a cart-pole with LQR and MPC. Both needed the equations of motion,
a linearization and a cost. This homework solves the same balance task **without a model**.
An agent only sees the observations and rewards that come back from `env.step(action)`,
and Proximal Policy Optimization (PPO) improves a neural-network policy from those samples.

We use [Gymnasium](https://gymnasium.farama.org/) `CartPole-v1`. Gymnasium is the maintained
successor of OpenAI Gym; code written for it still reads `import gymnasium as gym`.
PPO comes from [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (SB3).
You configure, train, evaluate and *question* PPO; you do not re-implement it.

| Part | Points | Student work |
|---|---:|---|
| A | 6 | Episode runner, hand-written baseline policy, evaluation over seeded episodes |
| B | 10 | Map PPO's mathematics onto SB3 arguments; train, evaluate and save a policy |
| C | 7 | Seeds, clip-range ablation and robustness tests; explain the evidence |
| D | 7 | PPO against LQR and MPC on the same CartPole-v1: linearize, design, compare |
| E | +5 | Domain randomization of the pole and a retrained, compared policy |

Do not edit TEST CELLs or their thresholds. **DO NOT USE A WHILE LOOP**: every episode loop must be a
bounded `for` loop. Training runs on the CPU and the whole notebook takes about six minutes.
Set `COURSE_HEADLESS=1` to skip only the optional animation.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import stable_baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from IPython.display import HTML, display
for root in (Path.cwd(), *Path.cwd().parents):
    if (root/'lecture8'/'rl_utils.py').exists():
        sys.path.insert(0, str(root/'lecture8')); break
else: raise FileNotFoundError('Start inside the course repository')
from rl_utils import (ENV_ID, SOLVED_RETURN, PPO_DEFAULTS, seed_everything, set_pole_parameters,
                      PerturbedCartPole, LearningCurveCallback, plot_learning_curves,
                      plot_robustness, animate_cartpole)
from cartpole_control import (MAX_FORCE, cartpole_step, ContinuousForceCartPole, continuous_training_env,
                              InitialStateWrapper, discrete_as_force, Timed, lqr_force_policy, CondensedMPC,
                              plot_controller_traces)
OUT = root/'lecture8'/'outputs'; OUT.mkdir(exist_ok=True)
HEADLESS = os.environ.get('COURSE_HEADLESS', '0') == '1'
seed_everything(0)
print(f'gymnasium {gym.__version__} | stable-baselines3 {stable_baselines3.__version__} | torch {torch.__version__}')

## From model-based control to reinforcement learning

A **Markov decision process** is the tuple $(\mathcal S,\mathcal A,p,r,\gamma)$: states, actions,
transition probabilities $p(s_{t+1}\mid s_t,a_t)$, a reward $r(s_t,a_t)$ and a discount $\gamma\in[0,1)$.
A policy $\pi(a\mid s)$ is a probability of acting; the goal is the largest expected discounted return
$$J(\pi)=\mathbb E_{\tau\sim\pi}\Big[\sum_{t=0}^{T-1}\gamma^t r_t\Big],\qquad
\tau=(s_0,a_0,r_0,s_1,a_1,r_1,\ldots).$$

| | Lectures 4 and 7 (LQR, MPC) | Lecture 8 (PPO) |
|---|---|---|
| What the designer supplies | $f(x,u)$, its linearization, $Q$, $R$ | a simulator to sample and a reward |
| Input | continuous wheel torque (Nm) | discrete push: 0 = left, 1 = right |
| Result | gain $K$ or a QP solved online | network weights found by trial and error |
| Cost of a design | one Riccati/QP solve, zero trials | tens of thousands of simulated steps |

Gymnasium's interface has two calls. `obs, info = env.reset(seed=s)` starts an episode, and
`obs, reward, terminated, truncated, info = env.step(action)` advances it by one step.
`terminated` means the task itself ended: the pole fell or the cart left the track.
`truncated` means a time limit cut the episode short, and the state was still fine.
The difference matters for learning, because after a truncation the future still has value.

**CartPole-v1.** The state is $s=[x,\dot x,\theta,\dot\theta]$, where $\theta>0$ leans the pole to the right.
The action pushes the cart with a force of $\pm10$ N for one 0.02 s step.
The reward is $+1$ for every step survived. An episode terminates when $|\theta|>12^\circ$ or $|x|>2.4$ m,
and it is truncated at 500 steps, so the best possible return is 500.
Gymnasium calls the task solved at an average return of **475**.

In [ ]:
env = gym.make(ENV_ID)
u = env.unwrapped
print('observation space:', env.observation_space)
print('action space     :', env.action_space)
print(f'cart {u.masscart} kg, pole {u.masspole} kg, pole half-length {u.length} m, '
      f'force ±{u.force_mag} N, dt {u.tau} s')
print(f'terminate at |theta| > {np.rad2deg(u.theta_threshold_radians):.0f} deg or |x| > {u.x_threshold} m; '
      f'truncate at {env.spec.max_episode_steps} steps')
frame_env = gym.make(ENV_ID, render_mode='rgb_array'); frame_env.reset(seed=0)
plt.figure(figsize=(5, 3.4)); plt.imshow(frame_env.render()); plt.axis('off')
plt.title('CartPole-v1 after reset(seed=0)'); plt.show(); frame_env.close()

## Part A: Episodes and a hand-written baseline (6 pts)

### A.1 Run one episode (3 pts)
A policy here is any function `policy(obs) -> action` with action in $\{0,1\}$.
`run_episode(env, policy, seed, max_steps=500)` resets with `seed`, then repeats
"ask the policy, step the environment" until `terminated` or `truncated`, or until `max_steps` actions.
The undiscounted return $G=\sum_t r_t$ is the score we report everywhere in this homework.

Return a dictionary with exactly these keys:

| key | content |
|---|---|
| `observations` | array of shape `(length+1, 4)`, from $s_0$ up to the last observation |
| `actions` | array of shape `(length,)`: integers here, forces in newtons in Part D |
| `rewards` | float array of shape `(length,)` |
| `episode_return` | `float`, the sum of the rewards |
| `length` | number of actions taken |
| `terminated`, `truncated` | the flags from the last `env.step` as `bool` (both `False` if `max_steps` stopped it) |

Call the policy exactly once per step and pass its output to `env.step` unchanged.
**DO NOT USE A WHILE LOOP**: a `for` loop over `range(max_steps)` with `break` is enough.

### A.2 A hand-written policy (1 pt)
Before learning anything, write a rule yourself. A linear test
$$a=\mathbb 1\big[k_x x+k_v\dot x+k_\theta\theta+k_\omega\dot\theta>0\big]$$
is a bang-bang version of the state feedback $u=-Kx$ from Lecture 4.
If the pole leans or rotates to the right, push right, so the cart moves back underneath it.
Choose the gains yourself. The test requires a mean return of at least 195 over 20 seeded episodes,
which is the old CartPole-v0 "solved" level.

### A.3 Average over seeded episodes (1 pt)
`episode_returns(env, policy, n_episodes, seed)` runs the episodes with seeds `seed, seed+1, …,
seed+n_episodes-1` and returns the array of returns. A single episode is not evidence; this function is
used for every comparison that follows.

In [ ]:
def run_episode(env, policy, seed, max_steps=500):
    """Run one episode with a bounded for loop; return the dictionary described above."""
    # TODO
    # 1. obs, _ = env.reset(seed=seed); start the observations list with obs
    # 2. for _ in range(max_steps): action = policy(obs); step; store obs, action, reward
    #    and break when terminated or truncated
    # 3. convert the lists to arrays and return the dictionary
    raise NotImplementedError("run_episode")

In [ ]:
def heuristic_policy(obs):
    """Return 1 (push right) or 0 (push left) from a linear test on [x, x_dot, theta, theta_dot]."""
    x, x_dot, theta, theta_dot = obs
    # TODO: choose gains so the cart moves under the falling pole
    raise NotImplementedError("heuristic_policy")

In [ ]:
def episode_returns(env, policy, n_episodes, seed):
    """Returns of run_episode for seeds seed, seed+1, ..., as an array of shape (n_episodes,)."""
    # TODO
    raise NotImplementedError("episode_returns")

In [ ]:
# TEST CELL (A): do not edit.
env = gym.make(ENV_ID)
def make_random_policy(seed):
    rng = np.random.default_rng(seed)
    return lambda obs: int(rng.integers(2))
ep = run_episode(env, make_random_policy(0), seed=3)
assert set(ep) == {'observations', 'actions', 'rewards', 'episode_return', 'length', 'terminated', 'truncated'}
n = ep['length']
assert ep['observations'].shape == (n+1, 4) and ep['actions'].shape == (n,) and ep['rewards'].shape == (n,)
np.testing.assert_array_equal(ep['observations'][0], env.reset(seed=3)[0])
assert ep['terminated'] is True and ep['truncated'] is False and ep['episode_return'] == n
again = run_episode(env, make_random_policy(0), seed=3)
np.testing.assert_array_equal(again['observations'], ep['observations'])
calls = []
ep0 = run_episode(env, lambda obs: calls.append(obs.copy()) or 0, seed=1)
assert len(calls) == ep0['length']
np.testing.assert_array_equal(np.array(calls), ep0['observations'][:-1])
short = run_episode(env, heuristic_policy, seed=0, max_steps=7)
assert short['length'] == 7 and not short['terminated'] and not short['truncated']
assert heuristic_policy(np.array([0, 0, .05, .1], np.float32)) == 1
assert heuristic_policy(np.array([0, 0, -.05, -.1], np.float32)) == 0
random_returns = episode_returns(env, make_random_policy(0), 20, seed=0)
heuristic_returns = episode_returns(env, heuristic_policy, 20, seed=0)
assert random_returns.shape == (20,) and 5 < random_returns.mean() < 60
assert heuristic_returns.mean() >= 195, heuristic_returns
print(f'random policy   : {random_returns.mean():6.1f} ± {random_returns.std():5.1f}')
print(f'heuristic policy: {heuristic_returns.mean():6.1f} ± {heuristic_returns.std():5.1f}')
print('Part A tests passed')

### Short answers (Part A, 1 pt)
1. List the four observation components and the two actions. Which component has no counterpart in
   Lecture 7's state `[s, theta, s_dot, theta_dot]`, and how do the inputs differ?
2. The reward is +1 every step and never mentions the angle. Why does maximizing the return still mean "keep
   the pole up"?
3. Why must a learning algorithm treat `terminated` and `truncated` differently?

**Write your answers here.**

## Part B: PPO with Stable-Baselines3 (10 pts)

### What PPO optimizes
The policy $\pi_\theta(a\mid s)$ is a small neural network (two hidden layers of 64 units) that outputs the
probabilities of the two actions. A second network $V_\phi(s)$ estimates the expected return from $s$.
The **policy gradient** pushes up the log-probability of actions that turned out better than expected:
$$\nabla_\theta J=\mathbb E\big[\nabla_\theta\log\pi_\theta(a_t\mid s_t)\,\hat A_t\big].$$

**Advantage with GAE.** "Better than expected" is measured by the advantage. Generalized Advantage Estimation
mixes one-step and long-horizon estimates through $\lambda$:
$$\delta_t=r_t+\gamma(1-d_t)V_\phi(s_{t+1})-V_\phi(s_t),\qquad
\hat A_t=\sum_{l\ge0}(\gamma\lambda)^l\,\delta_{t+l},$$
where $d_t=1$ only if the episode **terminated** at $t$. With $\lambda=0$ the estimate is the one-step TD error,
which has low variance but is biased by $V_\phi$. With $\lambda=1$ it is the Monte-Carlo return minus
$V_\phi(s_t)$, which is unbiased but noisy.

**Clipped surrogate.** One batch of data is reused for several gradient epochs. To stop the new policy from
moving too far from the policy that collected the data, PPO uses the probability ratio
$\rho_t(\theta)=\pi_\theta(a_t\mid s_t)/\pi_{\theta_{\text{old}}}(a_t\mid s_t)$ and clips it:
$$L^{\text{CLIP}}(\theta)=\mathbb E_t\Big[\min\big(\rho_t\hat A_t,\ \operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)\,\hat A_t\big)\Big].$$
Once $\rho_t$ leaves $[1-\epsilon,1+\epsilon]$ in the direction the advantage favours, the gradient is zero,
so there is no more reward for moving further. The complete loss that SB3 minimizes is
$$\mathcal L=-L^{\text{CLIP}}+c_1\,\mathbb E\big[(V_\phi(s_t)-\hat R_t)^2\big]-c_2\,\mathbb E\big[\mathcal H[\pi_\theta(\cdot\mid s_t)]\big],$$
with value target $\hat R_t=\hat A_t+V_\phi(s_t)$ and an entropy bonus $\mathcal H$ that keeps exploring.

**The loop.** Run $N$ environments for $T$ steps, compute $\hat A$, take $K$ epochs of minibatch Adam steps
on $\mathcal L$, and repeat. During training actions are **sampled** from $\pi_\theta$, which is how PPO
explores. For evaluation we take the most probable action (`deterministic=True`).

| Symbol | Meaning | SB3 argument | `PPO_DEFAULTS` |
|---|---|---|---|
| $N$ | parallel environments | `n_envs` of `make_vec_env` | 8 |
| $T$ | steps per environment per update | `n_steps` | 32 |
| — | minibatch size | `batch_size` | 256 |
| $K$ | epochs per update | `n_epochs` | 20 |
| $\gamma$ | discount | `gamma` | 0.98 |
| $\lambda$ | GAE parameter | `gae_lambda` | 0.8 |
| $\epsilon$ | clip range | `clip_range` | 0.2 → 0 |
| $\alpha$ | Adam step size | `learning_rate` | 1e-3 → 0 |
| $c_1$, $c_2$ | value and entropy weights | `vf_coef`, `ent_coef` | 0.5, 0.0 |

These are the tuned CartPole values of the RL Baselines3 Zoo. "→ 0" means a linear decay over training.
SB3 accepts a **schedule** for `learning_rate` and `clip_range`: a function of
`progress_remaining`, which goes from 1 at the start to 0 at the end of `learn`.
A linear decay from $v_0$ is therefore simply `lambda progress_remaining: v0 * progress_remaining`.

### What to implement
* **B.1 `make_training_env` (1 pt).** Use `make_vec_env(ENV_ID, n_envs=..., seed=..., wrapper_class=..., wrapper_kwargs=...)`.
  The wrapper arguments are unused until Parts D and E.
* **B.2 `build_ppo` (3 pts).** Merge `PPO_DEFAULTS` with `overrides` (overrides win). Remove `n_envs` and
  `linear_decay`, which are not PPO arguments. When `linear_decay` is true, turn `learning_rate` and
  `clip_range` into linear schedules, otherwise pass the constants. Build
  `PPO('MlpPolicy', env, ..., seed=seed, device='cpu', verbose=0)`. A network this small is faster on the CPU.
* **B.3 `ppo_policy` (1 pt).** Wrap a trained model as `policy(obs) -> int` using
  `model.predict(obs, deterministic=True)`, so Part A's functions can evaluate it.
* **B.4 `train_ppo` (3 pts).** Build the environment (with `n_envs` taken from `overrides` or the defaults) and
  the model, create a `LearningCurveCallback(eval_freq=eval_freq)`, call
  `model.learn(total_timesteps, callback=...)`, close the environment and return `(model, callback.curve)`.
* **Results (2 pts).** The trained agent passes TEST CELL B.3–B.4: a mean of at least 475 on 20 unseen
  episodes, and identical actions after saving and loading.

In [ ]:
def make_training_env(seed, n_envs=8, wrapper_class=None, wrapper_kwargs=None):
    """Vectorized, seeded CartPole-v1 training environment."""
    # TODO
    raise NotImplementedError("make_training_env")

In [ ]:
def build_ppo(env, seed, **overrides):
    """PPO('MlpPolicy', ...) configured from PPO_DEFAULTS updated by overrides."""
    # TODO
    # 1. params = {**PPO_DEFAULTS, **overrides}; drop 'n_envs'; pop 'linear_decay'
    # 2. pop learning_rate and clip_range; make linear schedules if linear_decay
    # 3. return PPO('MlpPolicy', env, ..., seed=seed, device='cpu', verbose=0, **params)
    raise NotImplementedError("build_ppo")

In [ ]:
def ppo_policy(model):
    """Return policy(obs) -> int that takes the model's deterministic action."""
    # TODO
    raise NotImplementedError("ppo_policy")

In [ ]:
def train_ppo(seed, total_timesteps=100_000, eval_freq=10_000, wrapper_class=None, wrapper_kwargs=None, **overrides):
    """Train one PPO agent; return (model, learning-curve dict)."""
    # TODO
    raise NotImplementedError("train_ppo")

In [ ]:
# TEST CELL (B.1-B.2): do not edit. Configuration only, no training yet.
venv = make_training_env(0)
assert venv.num_envs == 8
assert make_training_env(0, n_envs=2).num_envs == 2
m = build_ppo(venv, 0)
assert (m.n_steps, m.batch_size, m.n_epochs, m.n_envs) == (32, 256, 20, 8)
np.testing.assert_allclose([m.gamma, m.gae_lambda, m.ent_coef, m.vf_coef], [.98, .8, 0., .5])
np.testing.assert_allclose([m.lr_schedule(1.), m.lr_schedule(.5)], [1e-3, 5e-4])
np.testing.assert_allclose([m.clip_range(1.), m.clip_range(.25)], [.2, .05])
assert m.device.type == 'cpu' and m.seed == 0
m2 = build_ppo(venv, 0, clip_range=.5, gamma=.9, linear_decay=False)
np.testing.assert_allclose([m2.clip_range(.1), m2.lr_schedule(.1), m2.gamma], [.5, 1e-3, .9])
assert PPO_DEFAULTS['clip_range'] == .2 and PPO_DEFAULTS['linear_decay'] is True, 'do not modify PPO_DEFAULTS'
venv.close()
print('Part B configuration tests passed')

In [ ]:
# Given: train the default agent (about 20 s) and save it.
seed_everything(0)
model, curve = train_ppo(seed=0)
model.save(OUT/'ppo_cartpole_seed0')
plot_learning_curves({'PPO, seed 0': [curve]}); plt.show()
print('evaluated at steps:', curve['timesteps'])
print('mean return       :', np.round(curve['mean'], 1))

In [ ]:
# TEST CELL (B.3-B.4): do not edit.
eval_env = gym.make(ENV_ID)
ppo_returns = episode_returns(eval_env, ppo_policy(model), 20, seed=1000)
print(f'PPO: {ppo_returns.mean():.1f} ± {ppo_returns.std():.1f} over 20 unseen episodes')
assert curve['timesteps'][0] == 0 and curve['timesteps'][-1] >= 100_000
assert np.isfinite(curve['mean']).all() and curve['returns'].shape[1] == 5
assert ppo_returns.mean() >= SOLVED_RETURN, ppo_returns
loaded = PPO.load(OUT/'ppo_cartpole_seed0', device='cpu')
probe = np.random.default_rng(0).uniform(-.2, .2, size=(200, 4)).astype(np.float32)
np.testing.assert_array_equal(model.predict(probe, deterministic=True)[0], loaded.predict(probe, deterministic=True)[0])
assert ppo_policy(model)(probe[0]) in (0, 1)
print('Part B training tests passed')

In [ ]:
# Optional animation of one evaluation episode (skipped when COURSE_HEADLESS=1).
if not HEADLESS:
    episode = run_episode(eval_env, ppo_policy(model), seed=1000)
    display(HTML(animate_cartpole(episode['observations']).to_jshtml()))

## Part C: Evidence, not slogans (7 pts)

One training run is one sample from a random process. The network initialization, the sampled
actions and the episode starts all depend on the seed. The given drivers below call **your** functions to:

1. train the default configuration with three seeds (seed 0 is reused from Part B);
2. repeat the run with the clip range $\epsilon_0\in\{0.05,0.5\}$, three seeds each;
3. test the trained policies outside the training conditions. The pole length is scaled by
   $\{0.25,0.5,1,2,4,8\}$, and in a second test $\dot\theta$ is kicked once at step 100.
   Your Part A heuristic goes through the same tests.

These runs take about three minutes. Nothing here assumes that PPO wins, and some results may surprise you.
Report what the numbers show. As in Lecture 7, compare methods under identical seeds and episodes.

Scoring: 3 pts for complete experiments that pass the validity test, and 4 pts for the short answers.

In [ ]:
# Given experiment driver (about 3 minutes).
SEEDS = (0, 1, 2)
models = {0: model}
curves = {'clip 0.2 (default)': [curve]}
for seed in SEEDS[1:]:
    seed_everything(seed)
    models[seed], c = train_ppo(seed)
    curves['clip 0.2 (default)'].append(c)
for clip in (0.05, 0.5):
    curves[f'clip {clip}'] = []
    for seed in SEEDS:
        seed_everything(seed)
        curves[f'clip {clip}'].append(train_ppo(seed, clip_range=clip)[1])
plot_learning_curves({'default': curves['clip 0.2 (default)']}, 'Seed variation, default settings'); plt.show()
plot_learning_curves(curves, 'Clip-range ablation (3 seeds each)'); plt.show()

def first_solved(c):
    hit = np.flatnonzero(c['mean'] >= SOLVED_RETURN)
    return int(c['timesteps'][hit[0]]) if len(hit) else None
print(f'{"setting":20s} first evaluation >= 475 (steps) per seed   final mean per seed')
for label, cs in curves.items():
    print(f'{label:20s} {str([first_solved(c) for c in cs]):40s} {[float(c["mean"][-1]) for c in cs]}')

In [ ]:
# Given robustness driver: identical episodes for every policy.
LENGTH_SCALES = (0.25, 0.5, 1, 2, 4, 8)
def length_sweep(policy, n_episodes=10, seed=2000):
    return np.array([episode_returns(PerturbedCartPole(gym.make(ENV_ID), length_scale=s), policy, n_episodes, seed)
                     for s in LENGTH_SCALES])
policies = {'heuristic': heuristic_policy, **{f'PPO seed {s}': ppo_policy(m) for s, m in models.items()}}
robust = {name: length_sweep(p) for name, p in policies.items()}
plot_robustness(LENGTH_SCALES, robust, 'pole length / training pole length', 'Pole-length robustness'); plt.show()
print('mean return for pole-length scales:', LENGTH_SCALES)
for name, R in robust.items(): print(f'  {name:12s}', np.round(R.mean(axis=1), 1))

KICKS = (0.5, 1.0, 1.5, 2.0)  # rad/s added to theta_dot before step 100
kicked = {name: np.array([episode_returns(PerturbedCartPole(gym.make(ENV_ID), kick_step=100, kick=[0, 0, 0, k]),
                                          p, 10, 2000) for k in KICKS]) for name, p in policies.items()}
print('mean return for a theta_dot kick at step 100 (rad/s):', KICKS)
for name, R in kicked.items(): print(f'  {name:12s}', np.round(R.mean(axis=1), 1))

In [ ]:
# TEST CELL (C): validity only. There is deliberately no assertion about which method wins.
assert set(models) == set(SEEDS) and all(len(cs) == 3 for cs in curves.values())
t0 = curves['clip 0.2 (default)'][0]['timesteps']
for cs in curves.values():
    for c in cs:
        np.testing.assert_array_equal(c['timesteps'], t0)
        assert np.isfinite(c['mean']).all() and np.all((c['mean'] >= 0) & (c['mean'] <= 500))
for R in [*robust.values(), *kicked.values()]:
    assert R.shape in [(len(LENGTH_SCALES), 10), (len(KICKS), 10)] and np.isfinite(R).all()
assert gym.make(ENV_ID).unwrapped.length == .5, 'perturbations must not leak into new environments'
print('Part C validity tests passed')

### Short answers (Part C, 4 pts)
1. What does the clip in $L^{\text{CLIP}}$ prevent? On CartPole, how large was the effect of $\epsilon_0$ compared
   with the differences between seeds? What would you need before claiming that one $\epsilon_0$ is better?
2. Look at the step-0 point of every learning curve. Why do we evaluate with `deterministic=True` while training
   samples actions, and what does a high return from an untrained network tell you about this task and
   about five-episode evaluations?
3. Which perturbations broke the PPO policies, and which broke your heuristic? Explain the result using
   the states and parameters each policy had seen during training or design. Every PPO run scores 500 at the
   nominal length: why is that perfect training score not evidence of robustness?

**Write your answers here.**

## Part D: PPO versus LQR and MPC on the same cart-pole (7 pts)

So far PPO was compared only with a hand-written rule. Now compare it with the model-based controllers of
Lectures 4 and 7, **on exactly the same plant**. The given module `cartpole_control.py` provides four pieces.

* `cartpole_step(x, F)` reproduces one `CartPole-v1` step with a *continuous* force $F$ (N). With
  $M=m_c+m_p$ and $\ell$ the half-length of the pole,
  $$\ddot\theta=\frac{g\sin\theta-\cos\theta\,\dfrac{F+m_p\ell\dot\theta^2\sin\theta}{M}}
  {\ell\left(\tfrac43-\dfrac{m_p\cos^2\theta}{M}\right)},\qquad
  \ddot x=\frac{F+m_p\ell\,(\dot\theta^2\sin\theta-\ddot\theta\cos\theta)}{M},$$
  followed by an explicit Euler step with $\tau=0.02$ s: $x_{k+1}=x_k+\tau\dot x_k$, $\dot x_{k+1}=\dot x_k+\tau\ddot x_k$,
  and likewise for $\theta$. This is a *different* cart-pole from Lecture 7: a force-driven cart, not a wheel torque.
* `ContinuousForceCartPole` applies any $F\in[-10,10]$ N to the real Gymnasium environment, with the same
  integration, the same termination and the same 500-step limit. PPO's discrete push is the special case $F=\pm10$ N.
* `lqr_force_policy(A, B, Q, R)` is the clipped LQR you wrote in Lectures 4 and 7. It solves the discrete
  algebraic Riccati equation for $P$, forms $K=(R+B^TPB)^{-1}B^TPA$ and returns `(policy, K, P)` with
  `policy(obs) = float(clip(-K @ obs, -10, 10))`.
* `CondensedMPC` is a linear MPC with $|F|\le10$ N. Its predicted states are kept inside $|x|\le2.4$ m and
  $|\theta|\le10^\circ$ by penalized slacks. It eliminates the states ($X=\Phi x_0+\Gamma U$), unlike the sparse QP of HW6.

| Controller | Uses a model? | Action | What the designer supplied |
|---|---|---|---|
| heuristic (Part A) | no | $\pm10$ N | a sign rule from intuition |
| PPO discrete (Part B) | no | $\pm10$ N | reward, simulator, about $10^5$ samples |
| PPO continuous (trained below) | no | $F\in[-10,10]$ N | reward, simulator, about $10^5$ samples |
| LQR, clipped | linearization | $\operatorname{clip}(-Kx)$ | $A$, $B$, $Q$, $R$ |
| MPC, $N=50$ | linearization | QP solution | $A$, $B$, $Q$, $R$, $P$, $N$, limits |

### D.1 Linearize the discrete step (3 pts)
`cartpole_step` is already a discrete map $x_{k+1}=f(x_k,F_k)$, so no matrix exponential is needed. Linearize it
around the upright equilibrium with central differences:
$$A_{:,j}\approx\frac{f(x_0+\varepsilon e_j,u_0)-f(x_0-\varepsilon e_j,u_0)}{2\varepsilon},\qquad
B\approx\frac{f(x_0,u_0+\varepsilon)-f(x_0,u_0-\varepsilon)}{2\varepsilon}.$$
Return `A` with shape `(4, 4)` and `B` with shape `(4, 1)`. `step_fn(x, u)` takes a state array and a scalar force.

The driver below passes your $A$, $B$ to `lqr_force_policy`. The MPC reuses the LQR $P$ as its terminal weight,
so it reproduces LQR whenever no limit is active (Lecture 7).

### D.2 One cost for every controller (2 pts)
For an episode from `run_episode` on `ContinuousForceCartPole` (states $x_t$ = `observations[:-1]`, forces $F_t$ = `actions`),
return a dictionary with
`length`, `survived` (500 steps and not terminated), `quadratic_cost` $J=\sum_t x_t^TQx_t+F_t^TRF_t$,
`peak_force` $\max|F_t|$, `mean_abs_force` and `max_abs_theta_deg` (over all observations).
$J$ is the cost that LQR minimizes. Keep that in mind when you compare the numbers.

Scoring: 5 pts for D.1–D.2 and 2 pts for the short answers.

In [ ]:
# Given weights, shared by LQR, MPC and the cost metric.
Q_D = np.diag([1.0, 0.1, 10.0, 0.1])   # [x, x_dot, theta, theta_dot]
R_D = np.array([[0.01]])               # force in newtons

In [ ]:
def linearize_step(step_fn, x0, u0, eps=1e-6):
    """Central-difference Jacobians of x_next = step_fn(x, u): A (4x4), B (4x1)."""
    # TODO
    raise NotImplementedError("linearize_step")

In [ ]:
def controller_metrics(episode, Q, R):
    """length, survived, quadratic_cost, peak_force, mean_abs_force, max_abs_theta_deg."""
    # TODO
    raise NotImplementedError("controller_metrics")

In [ ]:
# TEST CELL (D.1-D.2): do not edit.
M_t = np.array([[1, 2, 0, 0], [0, 1, 3, 0], [0, 0, 1, 4], [5, 0, 0, 1.]]); b_t = np.array([1., 0, -2, 3])
A_t, B_t = linearize_step(lambda x, u: M_t @ x + b_t * float(u), np.ones(4), 2.0)
np.testing.assert_allclose(A_t, M_t, atol=1e-6); np.testing.assert_allclose(B_t, b_t[:, None], atol=1e-6)
A_cp, B_cp = linearize_step(cartpole_step, np.zeros(4), 0.0)
assert A_cp.shape == (4, 4) and B_cp.shape == (4, 1)
dx = np.array([.001, -.002, .003, .001])
np.testing.assert_allclose(cartpole_step(dx, .05), A_cp @ dx + B_cp[:, 0]*.05, atol=1e-7)
_, K_t, _ = lqr_force_policy(A_cp, B_cp, Q_D, R_D)
assert np.max(np.abs(np.linalg.eigvals(A_cp - B_cp@K_t))) < 1, 'your linearization must give a stabilizing LQR'
fake = dict(observations=np.array([[0, 0, .1, 0], [0, 0, -.2, 0], [0, 0, 0, 0]]), actions=np.array([1., -2.]),
            rewards=np.ones(2), episode_return=2., length=2, terminated=False, truncated=False)
m_t = controller_metrics(fake, Q_D, R_D)
np.testing.assert_allclose([m_t['quadratic_cost'], m_t['peak_force'], m_t['mean_abs_force'], m_t['max_abs_theta_deg']],
                           [.55, 2., 1.5, np.rad2deg(.2)])
assert m_t['length'] == 2 and m_t['survived'] is False
print('Part D function tests passed')

In [ ]:
# Given: model-based controllers from your linearization, and PPO with a continuous force (about 20 s).
A_lin, B_lin = linearize_step(cartpole_step, np.zeros(4), 0.0)
lqr, K_lqr, P_lqr = lqr_force_policy(A_lin, B_lin, Q_D, R_D, MAX_FORCE)
mpc = CondensedMPC(A_lin, B_lin, Q_D, R_D, P_lqr, N=50, state_limits=[2.4, np.inf, np.deg2rad(10), np.inf])
print('LQR gain K =', np.round(K_lqr, 2))

seed_everything(0)
cont_venv = make_training_env(0, wrapper_class=continuous_training_env)
ppo_cont = build_ppo(cont_venv, 0)   # same PPO_DEFAULTS; SB3 switches to a Gaussian policy for a Box action
cont_cb = LearningCurveCallback(eval_freq=10_000, make_env=lambda: continuous_training_env(gym.make(ENV_ID)))
ppo_cont.learn(100_000, callback=cont_cb); cont_venv.close()
plot_learning_curves({'PPO discrete ±10 N': [curve], 'PPO continuous F in [-10, 10] N': [cont_cb.curve]},
                     'Same PPO settings, two action spaces (seed 0)'); plt.show()

def continuous_ppo_force(model):
    return lambda obs: MAX_FORCE * float(np.clip(model.predict(obs, deterministic=True)[0][0], -1, 1))
CONTROLLERS = {'heuristic': discrete_as_force(heuristic_policy),
               'PPO discrete': discrete_as_force(ppo_policy(model)),
               'PPO continuous': continuous_ppo_force(ppo_cont),
               'LQR clipped': lqr,
               'MPC N=50': mpc}

def force_episode(policy, seed, wrap=None):
    """run_episode on ContinuousForceCartPole; a solver failure is reported, never hidden."""
    env_ = gym.make(ENV_ID) if wrap is None else wrap(gym.make(ENV_ID))
    try:
        return run_episode(ContinuousForceCartPole(env_), policy, seed)
    except RuntimeError as exc:
        print(f'  controller failed: {exc}')
        return None

In [ ]:
# Given comparison 1: nominal episodes (the training distribution of PPO), identical seeds.
nominal = {}
print(f'{"controller":15s} {"survived":>8s} {"mean J":>9s} {"mean |F|":>9s} {"peak |F|":>9s} {"max |theta|":>11s} {"ms/action":>9s}')
for name, ctrl in CONTROLLERS.items():
    timed = Timed(ctrl)
    episodes = [force_episode(timed, 1000 + i) for i in range(20)]
    ms = [controller_metrics(e, Q_D, R_D) for e in episodes if e is not None]
    nominal[name] = ms
    print(f'{name:15s} {sum(m["survived"] for m in ms):5d}/20 {np.mean([m["quadratic_cost"] for m in ms]):9.2f} '
          f'{np.mean([m["mean_abs_force"] for m in ms]):9.2f} {max(m["peak_force"] for m in ms):9.2f} '
          f'{max(m["max_abs_theta_deg"] for m in ms):9.2f} deg {timed.mean_ms:9.3f}')

# Given comparison 2: initial states far outside PPO's training starts (|s_i| <= 0.05).
INITIAL_STATES = {'x0 = 2.0 m': [2.0, 0, 0, 0], 'x0 = 2.2 m': [2.2, 0, 0, 0], 'theta0 = 0.2 rad': [0, 0, .2, 0],
                  'x_dot0 = 1.5 m/s': [0, 1.5, 0, 0], 'theta_dot0 = 1 rad/s': [0, 0, 0, 1.0],
                  'x0 = 1.5, theta0 = -0.15': [1.5, 0, -.15, 0]}
starts = {name: {label: force_episode(ctrl, 0, lambda e, s=s: InitialStateWrapper(e, s))
                 for label, s in INITIAL_STATES.items()} for name, ctrl in CONTROLLERS.items()}
print('\nsteps survived from each initial state (500 = whole episode)')
print(f'{"controller":15s}' + ''.join(f'{label:>26s}' for label in INITIAL_STATES))
for name, row in starts.items():
    print(f'{name:15s}' + ''.join(f'{(e["length"] if e else "failed"):>26}' for e in row.values()))
plot_controller_traces({name: row['x0 = 2.0 m'] for name, row in starts.items() if row['x0 = 2.0 m']},
                       title='Start at x0 = 2.0 m, pole upright (first 4 s)', t_max=4); plt.show()

In [ ]:
# Given comparison 3: the robustness tests of Part C for all five controllers.
def force_lengths(policy, n_episodes=10, seed=2000):
    return np.array([[np.nan if e is None else e['length'] for e in
                      (force_episode(policy, seed + i, lambda env_, s=s: PerturbedCartPole(env_, length_scale=s))
                       for i in range(n_episodes))] for s in LENGTH_SCALES], dtype=float)
force_robust = {name: force_lengths(ctrl) for name, ctrl in CONTROLLERS.items()}
plot_robustness(LENGTH_SCALES, force_robust, 'pole length / design length', 'All controllers, pole length'); plt.show()
print('mean steps survived for pole-length scales:', LENGTH_SCALES)
for name, R in force_robust.items(): print(f'  {name:15s}', np.round(np.nanmean(R, axis=1), 1))
force_kicked = {name: np.array([[np.nan if e is None else e['length'] for e in
                                 (force_episode(ctrl, 2000 + i, lambda env_, k=k: PerturbedCartPole(env_, kick_step=100, kick=[0, 0, 0, k]))
                                  for i in range(10))] for k in KICKS], dtype=float) for name, ctrl in CONTROLLERS.items()}
print('mean steps survived for a theta_dot kick at step 100 (rad/s):', KICKS)
for name, R in force_kicked.items(): print(f'  {name:15s}', np.round(np.nanmean(R, axis=1), 1))

In [ ]:
# TEST CELL (D): validity only. No assertion about which controller wins.
x_small = np.array([.01, 0., .02, 0.])
assert abs(mpc(x_small) - lqr(x_small)) < 1e-2, 'inactive limits + DARE terminal cost: MPC must equal LQR'
for name, ms in nominal.items():
    assert len(ms) == 20, f'{name} failed on nominal episodes'
    assert all(np.isfinite(m['quadratic_cost']) and 0 < m['length'] <= 500 and m['peak_force'] <= MAX_FORCE + 1e-6 for m in ms)
assert np.isfinite(cont_cb.curve['mean']).all() and set(starts) == set(CONTROLLERS)
for R in [*force_robust.values(), *force_kicked.values()]:
    assert R.shape in [(len(LENGTH_SCALES), 10), (len(KICKS), 10)]
print('Part D validity tests passed')

### Short answers (Part D, 2 pts)
1. What did each controller need before it could balance the pole: samples, a model, a linearization, tuning?
   Use the number of PPO training steps and the design inputs in the table above.
2. Rank the controllers by the nominal quadratic cost $J$ and by mean $|F|$. Why is this comparison biased toward
   LQR, and why does discrete PPO apply 10 N at every step? What would have to change in PPO's reward for it to care about $J$?
3. Use the initial-state table and the kick test. Where did MPC differ from clipped LQR, and why? Why does the MPC
   limit the predicted angle to $10^\circ$ rather than the $12^\circ$ termination angle?
4. None of the five controllers was designed for a longer pole. Which degraded most gracefully? What does that
   suggest about a fixed feedback structure compared with weights learned from one model?

**Write your answers here.**

## Part E (bonus): Domain randomization (+5)

Part C showed a policy that is perfect at one pole length and fails at another. **Domain randomization** trains
on a *distribution* of models instead of a single one:
$$\max_\theta\ \mathbb E_{p\sim\mathcal U}\big[J(\pi_\theta;p)\big],\qquad
p=(\ell,m_p),\quad \ell=s_\ell\,\ell_0,\ m_p=s_m\,m_{p,0},\quad s_\ell\sim\mathcal U(0.5,2),\ s_m\sim\mathcal U(0.5,2).$$

Implement `RandomizedPoleWrapper(gym.Wrapper)`:
* in `__init__`, store the ranges and the **nominal** length and mass of the wrapped environment;
* in `reset`, reset the wrapped environment first, then draw both scales with `self.unwrapped.np_random`.
  That generator is seeded by `reset(seed=...)`, so an episode is reproducible.
  Apply the scales to the *nominal* values with the given `set_pole_parameters`, so they never compound,
  and return `info` extended with `length_scale` and `masspole_scale`.

The given driver then trains three seeds with the wrapper through your `train_ppo(..., wrapper_class=...)`
and repeats the length sweep from Part C.

In [ ]:
class RandomizedPoleWrapper(gym.Wrapper):
    """Resample pole length and mass (relative to nominal) at every reset."""
    def __init__(self, env, length_range=(0.5, 2.0), masspole_range=(0.5, 2.0)):
        super().__init__(env)
        # TODO: store ranges and the nominal length / masspole
        raise NotImplementedError("RandomizedPoleWrapper")

    def reset(self, *, seed=None, options=None):
        # TODO: reset, sample with self.unwrapped.np_random, set_pole_parameters, extend info
        raise NotImplementedError("RandomizedPoleWrapper")

In [ ]:
# TEST CELL (E, BONUS): do not edit.
w = RandomizedPoleWrapper(gym.make(ENV_ID))
o1, i1 = w.reset(seed=5); L1 = w.unwrapped.length
o2, i2 = w.reset(seed=5)
assert L1 == w.unwrapped.length and i1['length_scale'] == i2['length_scale']
np.testing.assert_array_equal(o1, o2)
lengths = []
for s in range(60):
    _, info = w.reset(seed=s); u = w.unwrapped
    assert .5 <= info['length_scale'] <= 2 and .5 <= info['masspole_scale'] <= 2
    np.testing.assert_allclose([u.length, u.masspole], [.5*info['length_scale'], .1*info['masspole_scale']])
    np.testing.assert_allclose([u.total_mass, u.polemass_length], [u.masspole + u.masscart, u.masspole*u.length])
    lengths.append(u.length)
assert np.std(lengths) > .15
print('Bonus wrapper tests passed')

In [ ]:
# Given bonus experiment (about 1 minute): same seeds, same steps, randomized training.
dr_models, dr_curves = {}, []
for seed in SEEDS:
    seed_everything(seed)
    dr_models[seed], c = train_ppo(seed, wrapper_class=RandomizedPoleWrapper)
    dr_curves.append(c)
plot_learning_curves({'nominal training': curves['clip 0.2 (default)'], 'randomized training': dr_curves},
                     'Learning curves (evaluated on the nominal pole)'); plt.show()
dr_robust = {f'DR seed {s}': length_sweep(ppo_policy(m)) for s, m in dr_models.items()}
pooled = {'heuristic': robust['heuristic'],
          'PPO nominal (3 seeds)': np.hstack([robust[f'PPO seed {s}'] for s in SEEDS]),
          'PPO randomized (3 seeds)': np.hstack(list(dr_robust.values()))}
plot_robustness(LENGTH_SCALES, pooled, 'pole length / nominal length', 'Domain randomization, lengths 0.5-2 seen'); plt.show()
for name, R in {**{f'PPO seed {s}': robust[f'PPO seed {s}'] for s in SEEDS}, **dr_robust}.items():
    print(f'{name:12s}', np.round(R.mean(axis=1), 1))

### Short answers (Part E)
1. Inside the randomized range (0.5 to 2), did randomization help? Outside it (0.25, 4, 8)?
2. What did randomization cost: learning speed, nominal performance, or variance between seeds?
3. Name one physical parameter of a real cart-pole that you would randomize next, and explain why.

**Write your answers here.**

## Submission
Restart the kernel and run all cells. Submit the executed notebook and a PDF of it, including all plots,
printed tables and short answers. The bonus is optional. Do not submit `outputs/` model files unless asked.
See the README for the environment and for headless validation.